In [1]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error,r2_score)


In [2]:
df = pd.read_csv("D:\MlOps Day 1\data\data.csv")

df.head()

<>:1: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:1: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\tejal\AppData\Local\Temp\ipykernel_630544\2902469767.py:1: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  df = pd.read_csv("D:\MlOps Day 1\data\data.csv")


,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [3]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=56)

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [5]:
mlflow.set_experiment("Advertising Sales Prediction")

2026/09/05 11:34:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/05 11:34:21 INFO mlflow.store.db.utils: Updating database tables
2026/09/05 11:34:28 INFO mlflow.tracking.fluent: Experiment with name 'Advertising Sales Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/MlOps Day 1/notebooks/mlruns/1', creation_time=1788588268060, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788588268060, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

In [6]:
with mlflow.start_run(run_name="Linear Regression"):

    model = LinearRegression()
    
    model.fit(xtrain, ytrain)

    y_pred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

In [7]:
with mlflow.start_run(run_name="Ridge Regression"):

    model = Ridge(alpha=1.0)
    model.fit(xtrain, ytrain)

    ypred = model.predict(xtest)

    rmse = root_mean_squared_error(ytest, y_pred)
    r2 = r2_score(ytest, y_pred)

    # Log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    # Log the metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    # Log Artifacts
    mlflow.sklearn.log_model(sk_model = model, name="Ridge_Reg_Model")

In [8]:
# Turn on Scikit-learn autologging.
mlflow.sklearn.autolog()

In [9]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model.fit(xtrain, ytrain)

    test_pred = model.predict(xtest)

    test_rmse = root_mean_squared_error(ytest, test_pred) 
    test_mae = mean_absolute_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics
    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

2026/09/05 12:22:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [10]:
# For registering the model, we require the model URI.
# URI is a unique identifier for a model.

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/52cb594e234844599c4e06f276245177/model


In [11]:
registered_model = mlflow.register_model(
     model_uri=model_uri,
     name="Advertising_Sales_Model"
 )

registered_model

Successfully registered model 'Advertising_Sales_Model'.
2026/09/05 12:42:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 52cb594e234844599c4e06f276245177 has no artifacts at artifact path 'model', registering model based on models:/m-b1daf93ffa7f465ab3757b096c5dc109 instead
Created version '1' of model 'Advertising_Sales_Model'.


<ModelVersion: aliases=[], creation_timestamp=1788592353850, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788592353850, metrics=None, model_id=None, name='Advertising_Sales_Model', params=None, run_id='52cb594e234844599c4e06f276245177', run_link=None, source='models:/m-b1daf93ffa7f465ab3757b096c5dc109', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [12]:
from mlflow import MlflowClient

client = MlflowClient()

# 1. Assign an alias to a specific version
# (Sets the alias 'champion' to Version 2 of 'fraud_detector')
client.set_registered_model_alias(
    name="Advertising_Sales_Model", 
    alias="champion", 
    version="1"
)

In [13]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion" 
)

# Generate new predictions
new_data = pd.DataFrame({
    "TV": [150.0],
    "Radio": [25.0],
    "Newspaper": [30.0]
})

print(model.predict(new_data))

[14.2199203]
